<a href="https://colab.research.google.com/github/Sircheikh999/Examen_DataCollection/blob/main/books_to_scrape(Selenium).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installation de Selenium

In [1]:
!pip install google-colab-selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 94.9 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


# Importer packages

In [2]:
# importer packages
import pandas as pd
from selenium.webdriver.common.by import By
import google_colab_selenium as gs
import time

# Lancement du navigateur

In [3]:
# Lancer le navigateur
driver = gs.Chrome()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Scraping des données : Source 1 — Books to Scrape


In [4]:
url = 'https://books.toscrape.com/catalogue/page-1.html'
# ouvrir la page
driver.get(url)

In [22]:
df_final = pd.DataFrame()

# pages 1 à 50
for i in range(1, 51):

    url = f'https://books.toscrape.com/catalogue/page-{i}.html'

    # ouvrir la page
    driver.get(url)
    time.sleep(2)

    # containers = liens vers les produits
    containers = driver.find_elements(
        By.CSS_SELECTOR,
        'article.product_pod h3 a'
    )

    # nombre de produits sur la page
    number_of_products = len(containers)

    # récupérer tous les liens avant de changer de page
    product_urls = []

    for container in containers:
        product_urls.append(
            container.get_attribute('href')
        )

    data = []

    # parcourir les livres
    for product_url in product_urls:

        try:

            # ouvrir la page du livre
            driver.get(product_url)

            # récupérer les informations
            dic = {
                'page': i,

                'number_of_products': number_of_products,

                'title': driver.find_element(By.CSS_SELECTOR,'div.product_main h1').text,

                'price': driver.find_element(By.CSS_SELECTOR,'div.product_main p.price_color').text,

                'availability': driver.find_element(By.CSS_SELECTOR,'div.product_main p.instock.availability').text,

                'star_rating': driver.find_element(By.CSS_SELECTOR,'div.product_main p.star-rating').get_attribute('class'),

                'reviews': driver.find_element(By.CSS_SELECTOR,'table.table-striped tr:nth-child(7) td').text,

                'description': driver.find_element(By.CSS_SELECTOR,'#product_description + p').text,

                'product_type': driver.find_element(By.CSS_SELECTOR,'ul.breadcrumb li:nth-child(3) a').text,

                'tax': driver.find_element(By.CSS_SELECTOR,'table.table-striped tr:nth-child(5) td').text
            }

            data.append(dic)

        except:
            pass

    # dataframe de la page
    df = pd.DataFrame(data)

    # ajouter au dataframe final
    df_final = pd.concat(
        [df_final, df],
        axis=0
    ).reset_index(drop=True)

In [27]:
df_final.tail()

,page,number_of_products,title,price,availability,star_rating,reviews,description,product_type,tax
993,50,20,Beyond Good and Evil,£43.38,In stock (1 available),star-rating One,0,Friedrich Nietzsche's Beyond Good and Evil is ...,Philosophy,£0.00
994,50,20,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",£57.06,In stock (1 available),star-rating Four,0,High school student Kei Nagai is struck dead i...,Sequential Art,£0.00
995,50,20,A Spy's Devotion (The Regency Spies of London #1),£16.97,In stock (1 available),star-rating Five,0,"In England’s Regency era, manners and elegance...",Historical Fiction,£0.00
996,50,20,1st to Die (Women's Murder Club #1),£53.98,In stock (1 available),star-rating One,0,"James Patterson, bestselling author of the Ale...",Mystery,£0.00
997,50,20,"1,000 Places to See Before You Die",£26.08,In stock (1 available),star-rating Five,0,"Around the World, continent by continent, here...",Travel,£0.00


# Nettoyage des données : Source 1 — Books to Scrape

In [24]:
# localisons les valeurs manquantes
print(df_final.isna().sum())

page                  0
number_of_products    0
title                 0
price                 0
availability          0
star_rating           0
reviews               0
description           0
product_type          0
tax                   0
dtype: int64


In [25]:
# Vérification de doublon
print(df_final.duplicated().sum())

0


In [26]:
# Vérification finale
print(df_final.isna().any().sum())

0
